# Multi-Agent and Supervisor Patterns

A single agent with **many tools** works until tool lists grow, prompts blur, and the model picks the wrong capability. **Multi-agent** systems split responsibility: a **supervisor** routes work to **specialized workers** (docs, math, code), each with a narrow tool set and system prompt.

This notebook builds a **supervisor graph** over three **ReAct worker subgraphs** (`docs_agent`, `math_agent`, `code_agent`), uses **conditional edges** for routing, **handoff messages** between agents, a **parallel fan-out preview**, compares supervisor vs monolithic agent, and packages everything in **`build_supervisor_graph()`**.

Prerequisites: **04. Conditional Routing and Branching**, **06. Agent Loops and the ReAct Pattern**, **10. Subgraphs and Nested Workflows**.

In [ ]:
OPENAI_API_KEY = "sk-proj-placeholder-replace-with-your-real-key"

import json

import langchain_core

try:
    import langgraph
    print("langgraph:", langgraph.__version__)
except ImportError:
    print("langgraph: pip install langgraph")

print("langchain-core:", langchain_core.__version__)

---

## 1. Why Multi-Agent?

| Pain point (single agent) | Multi-agent fix |
|---------------------------|-----------------|
| 15+ tools confuse routing | Each worker has 1–3 tools |
| One bloated system prompt | Per-worker specialist prompts |
| Hard to test / observe | Route + worker traces are separate |
| Can't scale teams | Map roles to graph nodes |

**Supervisor pattern:** one node **decides who acts next**; workers **execute** and **return** control. The supervisor may loop until the task is complete or hand off to **`END`**.

---

## 2. LangGraph Multi-Agent Diagram

```
                    START
                      │
                      ▼
              ┌───────────────┐
              │  supervisor   │◄────────────────────────────┐
              │ (route task)  │                             │
              └───────┬───────┘                             │
                      │ next_agent                          │
         ┌────────────┼────────────┐                        │
         ▼            ▼            ▼                        │
   ┌──────────┐ ┌──────────┐ ┌──────────┐                   │
   │docs_agent│ │math_agent│ │code_agent│                   │
   │ (ReAct)  │ │ (ReAct)  │ │ (ReAct)  │                   │
   └────┬─────┘ └────┬─────┘ └────┬─────┘                   │
        │            │            │                          │
        └────────────┴────────────┴──────────────────────────┘
                      │
              done? ──┴── yes ──► END
```

Each **worker** is a **compiled subgraph** (manual ReAct loop or **`create_agent`**). The parent graph treats them as **nodes** that invoke the child graph.

---

## 3. Supervisor State Schema

Shared state carries the **conversation**, routing decision, and optional **task** metadata:

In [ ]:
from typing import Annotated, Literal
from typing_extensions import TypedDict

from langgraph.graph.message import add_messages


WorkerName = Literal["docs_agent", "math_agent", "code_agent", "FINISH"]


class SupervisorState(TypedDict):
    messages: Annotated[list, add_messages]
    next_agent: str          # routing key written by supervisor
    task: str                # original user goal (stable across handoffs)
    worker_turns: int        # cap loops supervisor ↔ workers

| Key | Reducer | Role |
|-----|---------|------|
| **`messages`** | `add_messages` | Full multi-agent transcript |
| **`next_agent`** | replace | Supervisor's routing decision |
| **`task`** | replace | User request preserved for workers |
| **`worker_turns`** | replace | Prevent infinite supervisor loops |

---

## 4. Notes API Corpus and Worker Tools

Same teaching corpus as **05** / **06** — each worker gets **only** the tools it needs:

In [ ]:
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)

DOC_CHUNKS = [
    {"id": "c1", "text": "The Notes API is built with FastAPI. Routes live under /notes with GET, POST, PUT, DELETE."},
    {"id": "c2", "text": "Alembic applies SQLAlchemy schema migrations. Run alembic upgrade head after pulling revisions."},
    {"id": "c3", "text": "JWT bearer tokens authenticate API requests. Send Authorization: Bearer token header."},
    {"id": "c4", "text": "Pytest fixtures share database setup. Use conftest.py for session-scoped engines."},
    {"id": "c5", "text": "Alembic autogenerate compares SQLAlchemy models to the live schema and drafts revision files."},
]


@tool
def search_docs(query: str) -> str:
    """Keyword search over Notes API documentation chunks."""
    tokens = set(query.lower().split())
    hits = []
    for chunk in DOC_CHUNKS:
        if tokens & set(chunk["text"].lower().split()):
            hits.append(f"[{chunk['id']}] {chunk['text']}")
    return "\n".join(hits) if hits else "No matching chunks."


@tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b


@tool
def multiply(a: int, b: int) -> int:
    """Multiply two integers."""
    return a * b


@tool
def describe_endpoint(method: str, path: str) -> str:
    """Describe a Notes API HTTP endpoint (teaching stub)."""
    catalog = {
        ("GET", "/notes"): "List all notes for the authenticated user.",
        ("POST", "/notes"): "Create a note; body requires title and content.",
        ("PUT", "/notes/{id}"): "Update an existing note by id.",
        ("DELETE", "/notes/{id}"): "Delete a note by id.",
    }
    return catalog.get((method.upper(), path), f"Unknown endpoint {method} {path}")

---

## 5. Worker Subgraphs — Compiled ReAct Agents

Each worker is a **small ReAct graph** (**06**). We compile once and invoke from parent nodes (**10**):

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langgraph.graph import END, START, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition


class WorkerState(TypedDict):
    messages: Annotated[list, add_messages]


def build_worker_agent(tools: list, system_prompt: str, name: str):
    """Compile a ReAct subgraph for one specialist."""
    worker_llm = llm.bind_tools(tools)

    def call_model(state: WorkerState) -> dict:
        msgs = [{"role": "system", "content": system_prompt}] + state["messages"]
        return {"messages": [worker_llm.invoke(msgs)]}

    builder = StateGraph(WorkerState)
    builder.add_node(f"{name}_model", call_model)
    builder.add_node(f"{name}_tools", ToolNode(tools))
    builder.add_edge(START, f"{name}_model")
    builder.add_conditional_edges(f"{name}_model", tools_condition)
    builder.add_edge(f"{name}_tools", f"{name}_model")
    return builder.compile()


docs_agent = build_worker_agent(
    [search_docs],
    "You are a documentation specialist for the FastAPI Notes API. Use search_docs. Cite chunk ids like [c2].",
    "docs",
)
math_agent = build_worker_agent(
    [add, multiply],
    "You are a math specialist. Use add and multiply tools for arithmetic. Show the result clearly.",
    "math",
)
code_agent = build_worker_agent(
    [describe_endpoint],
    "You are an API design specialist. Use describe_endpoint for HTTP route questions about /notes.",
    "code",
)

print("workers compiled:", type(docs_agent).__name__, type(math_agent).__name__, type(code_agent).__name__)

**Alternative:** swap `build_worker_agent` for **`create_agent(model=llm, tools=..., system_prompt=...)`** (**06**, section 12) — same topology, less boilerplate.

---

## 6. Handoff Messages Pattern

When the supervisor routes to a worker, inject a **handoff** so the worker knows **who sent it** and **what to focus on**:

In [ ]:
def make_handoff_message(from_agent: str, to_agent: str, task: str) -> SystemMessage:
    """Structured handoff the worker sees as context."""
    return SystemMessage(
        content=(
            f"[HANDOFF from {from_agent} to {to_agent}]\n"
            f"Task: {task}\n"
            f"Complete your specialty, then return a concise answer for the supervisor."
        )
    )


def make_worker_reply(agent_name: str, content: str) -> AIMessage:
    """Worker result tagged so supervisor can scan the transcript."""
    return AIMessage(content=f"[{agent_name}] {content}")


# Preview handoff shape
sample = make_handoff_message("supervisor", "docs_agent", "Explain JWT header format")
print(sample.content)

Handoffs can also use **`Command`** / **`Send`** APIs in advanced graphs; the **SystemMessage prefix** pattern is the clearest teaching default.

---

## 7. Supervisor Node — Decide `next_agent`

The supervisor reads **`task`** and recent **`messages`**, then writes **`next_agent`**. We use a **keyword router** for deterministic demos; production swaps in an LLM structured-output call:

In [ ]:
MAX_WORKER_TURNS = 6


def classify_next_agent(task: str, messages: list) -> WorkerName:
    """Keyword supervisor — replace with LLM router in production."""
    text = task.lower()
    # If a worker already answered, finish
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content.startswith("["):
            if any(tag in msg.content for tag in ("[docs_agent]", "[math_agent]", "[code_agent]")):
                return "FINISH"
    if any(w in text for w in ("add", "plus", "multiply", "sum", "calculate", "times")):
        return "math_agent"
    if any(w in text for w in ("get", "post", "put", "delete", "endpoint", "route", "/notes")):
        return "code_agent"
    if any(w in text for w in ("jwt", "alembic", "pytest", "fastapi", "migration", "doc", "token")):
        return "docs_agent"
    return "docs_agent"  # default specialist


def supervisor_node(state: SupervisorState) -> dict:
    turns = state.get("worker_turns", 0) + 1
    if turns > MAX_WORKER_TURNS:
        return {
            "next_agent": "FINISH",
            "worker_turns": turns,
            "messages": [AIMessage(content="[supervisor] Max worker turns reached.")],
        }
    nxt = classify_next_agent(state["task"], state["messages"])
    return {
        "next_agent": nxt,
        "worker_turns": turns,
        "messages": [AIMessage(content=f"[supervisor] Routing to {nxt}.")],
    }

---

## 8. Worker Wrapper Nodes — Invoke Subgraph, Return to Supervisor

Each parent **worker node** prepends a handoff, runs the subgraph, and appends a tagged reply:

In [ ]:
def _last_ai_text(messages: list) -> str:
    for msg in reversed(messages):
        if isinstance(msg, AIMessage) and msg.content and not msg.content.startswith("["):
            return msg.content if isinstance(msg.content, str) else str(msg.content)
    return "No answer produced."


def make_worker_node(agent_graph, agent_name: str):
    def worker_node(state: SupervisorState) -> dict:
        handoff = make_handoff_message("supervisor", agent_name, state["task"])
        worker_input = {
            "messages": [handoff, HumanMessage(content=state["task"])] + state["messages"],
        }
        result = agent_graph.invoke(worker_input)
        answer = _last_ai_text(result["messages"])
        return {"messages": [make_worker_reply(agent_name, answer)]}
    return worker_node


docs_worker = make_worker_node(docs_agent, "docs_agent")
math_worker = make_worker_node(math_agent, "math_agent")
code_worker = make_worker_node(code_agent, "code_agent")

---

## 9. Conditional Edges — Supervisor → Workers → Supervisor

Router reads **`next_agent`**; every worker edge returns to **`supervisor`**:

In [ ]:
def route_supervisor(state: SupervisorState) -> str:
    nxt = state.get("next_agent", "FINISH")
    if nxt == "FINISH":
        return END
    return nxt  # docs_agent | math_agent | code_agent


WORKER_PATH_MAP = {
    "docs_agent": "docs_agent",
    "math_agent": "math_agent",
    "code_agent": "code_agent",
}

---

## 10. Graph Factory — `build_supervisor_graph()`

Compile **once** at startup (**16**):

In [ ]:
def build_supervisor_graph():
    builder = StateGraph(SupervisorState)

    builder.add_node("supervisor", supervisor_node)
    builder.add_node("docs_agent", docs_worker)
    builder.add_node("math_agent", math_worker)
    builder.add_node("code_agent", code_worker)

    builder.add_edge(START, "supervisor")
    builder.add_conditional_edges("supervisor", route_supervisor, WORKER_PATH_MAP)

    # Workers always return to supervisor for re-evaluation or FINISH
    builder.add_edge("docs_agent", "supervisor")
    builder.add_edge("math_agent", "supervisor")
    builder.add_edge("code_agent", "supervisor")

    return builder.compile()


supervisor_graph = build_supervisor_graph()
print("nodes:", list(supervisor_graph.get_graph().nodes))

---

## 11. Invoke — Route to the Right Worker

Provide **`messages`**, **`task`**, and counters on first invoke:

In [ ]:
INITIAL = {"messages": [], "next_agent": "", "worker_turns": 0}


def run_supervisor(task: str) -> dict:
    return supervisor_graph.invoke({**INITIAL, "task": task, "messages": [HumanMessage(content=task)]})


def print_supervisor_trace(result: dict) -> None:
    for i, msg in enumerate(result["messages"]):
        preview = str(msg.content)[:90].replace("\n", " ")
        print(f"{i:02d} {msg.type:6s} {preview}")


for question in [
    "What Alembic command applies migrations?",
    "What is 17 times 4?",
    "Describe the POST /notes endpoint.",
]:
    out = run_supervisor(question)
    print(f"\n=== {question} ===")
    print_supervisor_trace(out)
    print("turns:", out["worker_turns"])

Trace pattern: **`human → supervisor (route) → worker (handoff + ReAct) → supervisor (FINISH) → END`**.

---

## 12. Parallel Workers Preview (Fan-Out)

Some tasks need **multiple specialists at once** (e.g. docs + math). LangGraph **`Send`** fans out to worker nodes in parallel; results merge via **`add_messages`**:

In [ ]:
from langgraph.types import Send


class FanOutState(TypedDict):
    task: str
    messages: Annotated[list, add_messages]


def fan_out_router(state: FanOutState):
    """Return multiple Send targets — preview only."""
    return [
        Send("docs_agent", {"task": state["task"], "messages": state["messages"]}),
        Send("math_agent", {"task": state["task"], "messages": state["messages"]}),
    ]


def fan_docs(state: FanOutState) -> dict:
    r = docs_agent.invoke({"messages": [HumanMessage(content=state["task"])]})
    return {"messages": [make_worker_reply("docs_agent", _last_ai_text(r["messages"]))]}


def fan_math(state: FanOutState) -> dict:
    r = math_agent.invoke({"messages": [HumanMessage(content="compute 3 times 11")]})
    return {"messages": [make_worker_reply("math_agent", _last_ai_text(r["messages"]))]}


def aggregate_node(state: FanOutState) -> dict:
    lines = [m.content for m in state["messages"] if isinstance(m, AIMessage)]
    return {"messages": [AIMessage(content="[aggregate] " + " | ".join(lines))]}


fan_builder = StateGraph(FanOutState)
fan_builder.add_node("docs_agent", fan_docs)
fan_builder.add_node("math_agent", fan_math)
fan_builder.add_node("aggregate", aggregate_node)
fan_builder.add_conditional_edges(START, fan_out_router)
fan_builder.add_edge("docs_agent", "aggregate")
fan_builder.add_edge("math_agent", "aggregate")
fan_builder.add_edge("aggregate", END)

fan_graph = fan_builder.compile()
fan_result = fan_graph.invoke({
    "task": "JWT header and also 3 times 11",
    "messages": [HumanMessage(content="JWT header and also 3 times 11")],
})
print(fan_result["messages"][-1].content)

Production fan-out adds **synchronization**, **timeouts**, and **reducers** beyond `add_messages`. See LangGraph map-reduce docs for full patterns.

---

## 13. Visualize the Supervisor Graph

Export Mermaid for architecture docs (**03**):

In [ ]:
try:
    mermaid = supervisor_graph.get_graph().draw_mermaid()
    print(mermaid[:900], "..." if len(mermaid) > 900 else "")
except Exception as exc:
    print("Mermaid export:", exc)

---

## 14. Supervisor vs Single Agent with Many Tools

A **monolithic** agent binds **all** tools — simpler wiring, worse routing at scale:

In [ ]:
from langchain.agents import create_agent

all_tools = [search_docs, add, multiply, describe_endpoint]
mono_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt="You handle docs, math, and API endpoints. Pick the right tool.",
)

mono = mono_agent.invoke({"messages": [HumanMessage(content="Alembic upgrade command?")]})
print("monolithic answer:", _last_ai_text(mono["messages"]))

multi = run_supervisor("Alembic upgrade command?")
print("supervisor last:", multi["messages"][-1].content[:200])

| Aspect | **Supervisor + workers** | **Single agent, all tools** |
|--------|--------------------------|----------------------------|
| Tool selection | Narrow per worker | Model chooses among many |
| Prompt focus | Specialist system prompts | One combined prompt |
| Observability | Per-node traces | Single agent trace |
| Complexity | Higher graph wiring | Lower upfront cost |
| Best when | Teams, 5+ tools, strict roles | Prototypes, &lt;5 tools |

Start monolithic; **graduate to supervisor** when tool errors or prompt bleed appear.

---

## 15. `create_agent` Workers (Alternative)

Swap **`build_worker_agent`** for factory agents when you do not need custom inner nodes:

In [ ]:
docs_agent_factory = create_agent(llm, [search_docs], system_prompt="Docs specialist. Cite [c#].")
factory_result = docs_agent_factory.invoke({
    "messages": [HumanMessage(content="JWT bearer header format?")],
})
print("factory worker:", _last_ai_text(factory_result["messages"]))

---

## 16. Design Guidelines

| Guideline | Why |
|-----------|-----|
| **Cap `worker_turns`** | Supervisor ↔ worker loops can churn |
| **Tag worker replies** | Supervisor scans `[docs_agent]` prefixes |
| **Handoff SystemMessage** | Workers get explicit task context |
| **Compile subgraphs once** | Workers are reusable compiled graphs |
| **Thin supervisor router** | Classification separate from execution |
| **Workers return to supervisor** | Enables multi-step delegation |
| **LLM router in production** | Keyword demo is for teaching only |

---

## 17. Common Mistakes

| Mistake | Consequence | Fix |
|---------|-------------|-----|
| Worker edge goes to `END` directly | Supervisor never re-evaluates | `worker → supervisor` |
| Shared tools on every worker | Same confusion as monolith | Split tool lists |
| No handoff context | Worker lacks task focus | `make_handoff_message` |
| Re-invoke subgraph with stale messages | Duplicate tool calls | Pass minimal worker input |
| `next_agent` typo in path map | Runtime routing error | Use `Literal` + map keys |
| Infinite supervisor loop | API cost spike | `MAX_WORKER_TURNS` |
| Rebuild parent graph per request | Slow cold start | `build_supervisor_graph()` once |

---

## 18. Summary

You built a **supervisor multi-agent graph**: **`supervisor`** routes via **`next_agent`** conditional edges to **`docs_agent`**, **`math_agent`**, and **`code_agent`** ReAct subgraphs; workers use **handoff messages**, return to the supervisor, and finish when a tagged reply is detected.

Key takeaways:

- **Supervisor state** = `messages` + `next_agent` + `task` + turn counter.
- **Workers** are **compiled subgraphs** invoked inside parent nodes (**10**).
- **Conditional edges** implement routing; **fixed return edges** close the loop.
- **`Send`** previews **parallel fan-out** for multi-specialist tasks.
- Prefer **supervisor** over **monolithic many-tool** agents when roles diverge.
- **`build_supervisor_graph()`** compiles once for production (**16**).

Next: **12. Agentic RAG with LangGraph** — combine **05**'s retrieve-grade-generate graph with **06**'s tool agent for hybrid RAG.